In [72]:
import pandas as pd
import numpy as np
import joblib

In [73]:
pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [74]:
df=pd.read_csv("crop_recommendation.csv")
df.head()

,Soil_Type,Soil_pH,Moisture_Level,Season,Location_Region,Previous_Crop,Recommended_Crop
0,Loamy,4.57,Low,Zaid,Tamil Nadu,Rice,Banana
1,Laterite,6.88,High,Winter,Karnataka,Wheat,Rice
2,Clay,7.15,High,Rabi,Uttar Pradesh,NaN,Cotton
3,Laterite,5.89,Low,Kharif,Karnataka,Groundnut,Cumin
4,Alluvial,6.76,High,Rabi,Gujarat,Sugarcane,Pulses


In [75]:
df.shape

(10000, 7)

In [76]:
df.columns

Index(['Soil_Type', 'Soil_pH', 'Moisture_Level', 'Season', 'Location_Region',
       'Previous_Crop', 'Recommended_Crop'],
      dtype='object')

In [77]:
df.dtypes

Soil_Type            object
Soil_pH             float64
Moisture_Level       object
Season               object
Location_Region      object
Previous_Crop        object
Recommended_Crop     object
dtype: object

In [78]:
df.isnull().sum()

Soil_Type              0
Soil_pH                0
Moisture_Level         0
Season                 0
Location_Region        0
Previous_Crop       1224
Recommended_Crop       0
dtype: int64

In [79]:
df["Previous_Crop"] = df["Previous_Crop"].fillna("None")
df["Soil_pH"] = df["Soil_pH"].fillna(df["Soil_pH"].median())


In [80]:
df["Previous_Crop"].isnull().sum()


np.int64(0)

In [81]:
df['Soil_pH'].describe()

count    10000.000000
mean         6.499623
std          1.150863
min          4.500000
25%          5.510000
50%          6.510000
75%          7.490000
max          8.500000
Name: Soil_pH, dtype: float64

In [82]:
for col in df.columns:
    if df[col].dtype=='object':
        df[col]=df[col].str.strip().str.title()

In [83]:
for col in df.columns:
    print(col," : ",df[col].unique())
    print()

Soil_Type  :  ['Loamy' 'Laterite' 'Clay' 'Alluvial' 'Red' 'Sandy' 'Black']

Soil_pH  :  [4.57 6.88 7.15 5.89 6.76 8.11 6.92 5.7  8.13 7.91 5.18 7.99 7.11 5.08
 7.44 5.95 7.23 5.12 6.2  4.74 7.32 6.69 5.5  7.72 8.44 8.28 4.73 6.03
 8.08 4.85 4.95 4.53 4.81 6.12 7.49 8.07 6.13 5.39 6.27 6.32 8.17 6.54
 7.33 6.23 6.67 6.77 7.83 8.02 7.05 6.35 7.46 6.5  7.73 8.4  7.78 4.75
 7.09 7.03 7.53 7.65 7.31 8.31 7.9  5.94 7.16 6.24 5.73 6.36 4.96 6.72
 5.35 7.62 6.09 7.21 7.02 5.98 4.84 7.29 6.46 6.64 6.15 6.51 6.38 8.23
 7.24 5.21 6.56 5.63 7.6  7.55 5.83 8.01 5.62 7.88 4.86 6.41 6.83 4.88
 4.61 7.63 7.86 5.8  7.58 8.35 6.91 7.12 6.04 4.93 6.9  7.85 6.78 5.27
 7.3  6.65 5.84 7.04 7.64 4.8  6.1  6.21 5.79 7.2  5.02 4.56 4.91 6.44
 6.96 5.41 4.82 6.84 4.7  7.42 5.85 4.79 6.33 4.72 6.01 6.89 6.3  7.28
 5.26 6.14 5.69 4.76 8.06 5.01 5.3  4.69 8.04 8.14 8.34 5.54 5.51 5.28
 4.55 7.47 8.49 7.52 6.79 5.64 5.66 5.59 7.   5.86 6.99 6.18 6.48 7.59
 5.1  5.96 6.11 8.3  7.54 6.93 8.18 7.1  7.96 7.77 7.07 7.01

In [84]:
season_map = {
    "Winter": "Rabi",
    "Summer": "Zaid"
}

df["Season"] = df["Season"].replace(season_map)


In [85]:
df['Recommended_Crop'].value_counts()

Recommended_Crop
Maize        1054
Pulses       1014
Rice         1011
Cotton       1010
Sugarcane    1007
Banana       1004
Soybean      1001
Cumin         982
Wheat         965
Groundnut     952
Name: count, dtype: int64

In [86]:
df.head()

,Soil_Type,Soil_pH,Moisture_Level,Season,Location_Region,Previous_Crop,Recommended_Crop
0,Loamy,4.57,Low,Zaid,Tamil Nadu,Rice,Banana
1,Laterite,6.88,High,Rabi,Karnataka,Wheat,Rice
2,Clay,7.15,High,Rabi,Uttar Pradesh,None,Cotton
3,Laterite,5.89,Low,Kharif,Karnataka,Groundnut,Cumin
4,Alluvial,6.76,High,Rabi,Gujarat,Sugarcane,Pulses


In [87]:
df.to_csv("crop_data_cleaned.csv", index=False)

## Label Encoding

In [88]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier


In [89]:
cat_cols = [
    "Soil_Type",
    "Moisture_Level",
    "Season",
    "Location_Region",
    "Previous_Crop"
]

num_cols = ["Soil_pH"]


In [90]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)


In [91]:
model = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])


In [92]:
x=df.drop("Recommended_Crop",axis=1)
y=df["Recommended_Crop"]

In [93]:
x = x.dropna()
# y not updated ❌


In [94]:
x_train, x_test, y_train, y_test = train_test_split(x, y)

x_train = x_train.dropna()  # ❌


In [95]:
df = pd.concat([x, y], axis=1)


In [96]:
df["Previous_Crop"] = df["Previous_Crop"].fillna("None")


In [97]:
x = df.drop("Recommended_Crop", axis=1)
y = df["Recommended_Crop"]

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [98]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)


(8000, 15)
(8000,)
(2000, 15)
(2000,)


In [99]:
print(df.columns.tolist())


['Soil_Type', 'Soil_pH', 'Moisture_Level', 'Season', 'Location_Region', 'Previous_Crop', 'Recommended_Crop']


In [100]:
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
probs = model.predict_proba(x_test)


In [101]:
from sklearn.preprocessing import OneHotEncoder

ohe_cols = ["Soil_Type", "Moisture_Level", "Season"]

x_ohe = pd.get_dummies(x[ohe_cols])

In [102]:
from sklearn.preprocessing import LabelEncoder

le_prev = LabelEncoder()
x["Previous_Crop"] = le_prev.fit_transform(x["Previous_Crop"])

X_final = pd.concat(
    [x_ohe, x[["Soil_pH", "Previous_Crop"]]],
    axis=1
)


In [103]:
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

In [104]:
x.head()

,Soil_Type,Soil_pH,Moisture_Level,Season,Location_Region,Previous_Crop
0,Loamy,4.57,Low,Zaid,Tamil Nadu,5
1,Laterite,6.88,High,Rabi,Karnataka,7
2,Clay,7.15,High,Rabi,Uttar Pradesh,3
3,Laterite,5.89,Low,Kharif,Karnataka,1
4,Alluvial,6.76,High,Rabi,Gujarat,6


In [105]:
x.shape,y.shape

((10000, 6), (10000,))

In [106]:
import joblib

joblib.dump(feature_encoders,"feature_encoders.pkl")
joblib.dump(target_encoder,"target_encoder.pkl")

['target_encoder.pkl']

## Train and Test


In [112]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier


In [113]:
df = pd.read_csv("crop_data_cleaned.csv")   


In [114]:
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
      .str.replace("/", "_")
)


In [115]:
print(df.columns.tolist())


['Soil_Type', 'Soil_pH', 'Moisture_Level', 'Season', 'Location_Region', 'Previous_Crop', 'Recommended_Crop']


In [130]:
pip install catboost

  Using cached catboost-1.2.8-cp312-cp312-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached narwhals-2.15.0-py3-none-any.whl.metadata (13 kB)
Using cached catboost-1.2.8-cp312-cp312-win_amd64.whl (102.4 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.9 MB 4.2 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.9 MB 5.6 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.9 MB 4.1 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 4.6 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.9 MB 4.4 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.9 MB 4.4 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.9 MB 4.4 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.9 MB 3.4 MB/s eta 0:00:02
   -

In [116]:
df["Previous_Crop"] = df["Previous_Crop"].fillna("None")


In [117]:
df.isnull().sum()


Soil_Type           0
Soil_pH             0
Moisture_Level      0
Season              0
Location_Region     0
Previous_Crop       0
Recommended_Crop    0
dtype: int64

In [118]:
X = df.drop("Recommended_Crop", axis=1)
y = df["Recommended_Crop"]


In [124]:
categorical_cols = [
    "Soil_Type",
    "Moisture_Level",
    "Season",
    "Location_Region",
    "Previous_Crop"
]

numerical_cols = ["Soil_pH"]


In [125]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)


In [126]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [127]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        class_weight="balanced"
    ))
])


In [128]:
model.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [129]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.107
              precision    recall  f1-score   support

      Banana       0.10      0.10      0.10       201
      Cotton       0.12      0.12      0.12       202
       Cumin       0.08      0.08      0.08       197
   Groundnut       0.14      0.13      0.13       190
       Maize       0.10      0.10      0.10       211
      Pulses       0.12      0.11      0.12       203
        Rice       0.11      0.11      0.11       202
     Soybean       0.15      0.15      0.15       200
   Sugarcane       0.09      0.09      0.09       201
       Wheat       0.07      0.07      0.07       193

    accuracy                           0.11      2000
   macro avg       0.11      0.11      0.11      2000
weighted avg       0.11      0.11      0.11      2000



In [131]:
cat_features = [
    X.columns.get_loc(col)
    for col in X.columns
    if X[col].dtype == "object"
]


In [132]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)


0:	learn: 0.1101250	total: 619ms	remaining: 5m 8s
100:	learn: 0.4760000	total: 51.5s	remaining: 3m 23s
200:	learn: 0.8200000	total: 1m 50s	remaining: 2m 43s
300:	learn: 0.9601250	total: 2m 47s	remaining: 1m 50s
400:	learn: 0.9926250	total: 4m 3s	remaining: 1m
499:	learn: 0.9973750	total: 5m 5s	remaining: 0us


In [133]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.086


In [134]:
import numpy as np

probs = model.predict_proba(X_test)

top3 = np.argsort(probs, axis=1)[:, -3:]

labels = model.classes_

top3_labels = labels[top3]

top3_acc = np.mean([
    y_test.iloc[i] in top3_labels[i]
    for i in range(len(y_test))
])

print("Top-3 Accuracy:", top3_acc)


Top-3 Accuracy: 0.272
